In [1]:
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 105.7 MB/s eta 0:00:0000:01
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92


In [4]:
import os
import yaml
from roboflow import Roboflow

rf = Roboflow(api_key="O0uBrl31Gd2SQ0zW60TL")
project = rf.workspace("ankit-gupta-kiopm").project("wagon-ocr")
dataset = project.version(3).download("yolov8")

# Resolve dataset directory directly using the download object
dataset_dir = os.path.abspath(dataset.location)
yaml_path = os.path.join(dataset_dir, "data.yaml")

with open(yaml_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Update paths to absolute Kaggle workspace paths
config['path'] = dataset_dir
config['train'] = "train/images"
config['val'] = "valid/images"
config['test'] = "test/images"

with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f)

print(f"Successfully sanitized dataset data.yaml at: {yaml_path}")

loading Roboflow workspace...
loading Roboflow project...
Successfully sanitized dataset data.yaml at: /kaggle/working/Wagon-OCR-3/data.yaml


In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11m.pt")
model.train(
    data=yaml_path,
    epochs=100,
    imgsz=1024,
    batch=16,
    device=0,
    optimizer="AdamW",
    cos_lr=True,
    box=8.5,
    cls=1.0,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.0,
    flipud=0.0,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    patience=15,
    workers=4,
    project="wagon_ocr_kaggle",
    name="yolov11m_v3"
)

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=8.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.0, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/Wagon-OCR-3/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11m_v3, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam

In [ ]:
import shutil
src_weights = "wagon_ocr_kaggle/yolov11m_v3/weights/best.pt"
dst_weights = "/kaggle/working/yolov11m_v3_best.pt"
if os.path.exists(src_weights):
    shutil.copy(src_weights, dst_weights)
    print("🎉 SUCCESS! Copied trained model weights.")